# Inferencia textual multilingüe con MiniLM

Dada una premisa y una hipótesis, predecimos si la premisa **implica** la hipótesis (0), **no basta para decidir** (neutralidad, 1) o **la contradice** (2).

Usamos el corpus oficial de _Contradictory, My Dear Watson_: 12.120 pares etiquetados de entrenamiento en 15 idiomas y 5.195 pares de test. Conservamos el texto original. El test aporta los textos e IDs de la entrega. Su plantilla fija el orden del CSV.

**Resultado local del voto MLP + XGBoost:** 62,14 % de accuracy y 61,84 % de macro-F1 en validación agrupada por premisa.


## Preparación

En Kaggle adjuntamos la competición, el código del proyecto y el checkpoint de MiniLM como tres recursos. La siguiente celda localiza los archivos y prepara los imports. El código puede venir en un ZIP o en una carpeta extraída.


In [ ]:
import hashlib
import json
import sys
from importlib.metadata import version
from pathlib import Path, PurePosixPath
from shutil import copyfileobj
from tempfile import TemporaryDirectory
from zipfile import BadZipFile, ZipFile


SOURCE_FILES = (
    "classifiers.py", "training.py", "minilm_submission.py", "replay_dense_rf.py",
    "pos_features.py", "preprocessing.py", "preprocesser.py", "relation_features.py",
    "vectorizer.py", "results_dense_generalization/protocol.json",
)


def find_source_root(inputs, workspace):
    """Encuentra una sola raíz completa entre carpetas y ZIP adjuntos."""
    module_names = {name for name in SOURCE_FILES if name.endswith(".py")}
    paths = list(inputs.rglob("*"))
    matches = {}
    incomplete = []

    for folder in sorted({p.parent for p in paths if p.name in module_names and p.is_file()}):
        missing = [name for name in SOURCE_FILES if not (folder / name).is_file()]
        if missing:
            incomplete.append(f"{folder}: faltan {', '.join(missing)}")
        else:
            matches[str(folder)] = (folder, None)

    for path in sorted(p for p in paths if p.suffix.lower() == ".zip" and p.is_file()):
        try:
            with ZipFile(path) as archive:
                files = {PurePosixPath(m.filename): m.filename for m in archive.infolist() if not m.is_dir()}
        except BadZipFile:
            incomplete.append(f"{path}: ZIP no válido")
            continue

        roots = sorted({p.parent for p in files if p.name in module_names})
        if not roots:
            incomplete.append(f"{path}: sin módulos del proyecto")
        for root in roots:
            location = f"{path}!/{root}"
            missing = [name for name in SOURCE_FILES if root / name not in files]
            if missing:
                incomplete.append(f"{location}: faltan {', '.join(missing)}")
            else:
                matches[location] = (path, {name: files[root / name] for name in SOURCE_FILES})

    if len(matches) > 1:
        raise ValueError(f"Hay varias raíces de código en {inputs}: " + "; ".join(matches))
    if not matches:
        details = "; ".join(incomplete) or "no se encontraron módulos ni archivos ZIP"
        raise FileNotFoundError(
            f"No se encontró una raíz de código completa en {inputs}. "
            f"Archivos requeridos: {', '.join(SOURCE_FILES)}. Búsqueda: {details}."
        )

    source, members = next(iter(matches.values()))
    if members is None:
        return source
    with ZipFile(source) as archive:
        for name, member_name in members.items():
            target = workspace / name
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member_name) as member, target.open("wb") as out:
                copyfileobj(member, out)
    return workspace


KAGGLE_INPUT = Path("/kaggle/input")
ON_KAGGLE = KAGGLE_INPUT.is_dir()
PROJECT = Path.cwd()
DATA_DIR = PROJECT / "data"
MODEL_DIR = PROJECT / ".local" / "minilm-checkpoint"
OUTPUT_DIR = PROJECT / ".local" / "minilm_submission"

if ON_KAGGLE:
    OUTPUT_DIR = Path("/kaggle/working")
    source_workspace = TemporaryDirectory()
    PROJECT = find_source_root(KAGGLE_INPUT, Path(source_workspace.name))

    datasets = [p.parent for p in KAGGLE_INPUT.rglob("sample_submission.csv")
                if (p.parent / "train.csv").is_file() and (p.parent / "test.csv").is_file()]
    checkpoints = sorted({
        p.parent for name in ("model.safetensors", "pytorch_model.bin")
        for p in KAGGLE_INPUT.rglob(name)
        if (p.parent / "modules.json").is_file()
    })
    if len(datasets) != 1:
        raise FileNotFoundError("Adjunta los datos oficiales de la competición.")
    if not checkpoints:
        raise FileNotFoundError("Adjunta el checkpoint MiniLM como un dataset separado en Add Input.")
    if len(checkpoints) > 1:
        raise ValueError("Adjunta un solo checkpoint MiniLM: " + ", ".join(map(str, checkpoints)))
    DATA_DIR, MODEL_DIR = datasets[0], checkpoints[0]

sys.path.insert(0, str(PROJECT))
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from threadpoolctl import threadpool_limits

from classifiers import build_classifier
from minilm_submission import FIXED_TOKENIZER, TEST_ROWS, prepare_checkpoint, validate_submission, verify_official_data
from replay_dense_rf import _full_features
from training import write_submission

## Representación de los textos

MiniLM es un encoder multilingüe preentrenado. Lo mantenemos congelado para obtener, en cada frase, un vector de 384 componentes con pooling por media y normalización L2. Verificamos que se carga el tokenizador Unigram y que el máximo es de 128 tokens.

Para cada par concatenamos `[p, h, |p − h|, p × h]`, con `p` para premisa y `h` para hipótesis. Los primeros dos bloques conservan las frases por separado. Los otros dos comparan cada componente mediante diferencia absoluta y producto. El clasificador recibe 1.536 características. Codificamos una sola vez cada frase distinta y reconstruimos el orden original de los pares.


In [ ]:
verify_official_data(DATA_DIR)
options = {"dtype": {"id": str}, "keep_default_na": False}
train = pd.read_csv(DATA_DIR / "train.csv", **options)
test = pd.read_csv(DATA_DIR / "test.csv", **options)
sample = pd.read_csv(DATA_DIR / "sample_submission.csv", **options)
if (len(train) != 12120 or len(test) != TEST_ROWS or len(sample) != TEST_ROWS
        or not all(frame.id.is_unique for frame in (train, test, sample))
        or set(test.id) != set(sample.id) or set(train.label) != {0, 1, 2}):
    raise ValueError("Las filas, los IDs o las etiquetas oficiales no coinciden.")
test = test.set_index("id").loc[sample.id].reset_index()

protocol = json.loads((PROJECT / "results_dense_generalization/protocol.json").read_text(encoding="utf-8"))

# Recuperamos las posiciones de cada frase para reconstruir los pares.
rows = pd.concat([train, test], ignore_index=True)
texts = pd.concat([rows.premise, rows.hypothesis], ignore_index=True)
codes, unique = pd.factorize(texts)

# Usamos CPU como en la ejecución de Kaggle con el tokenizador corregido.
torch.set_num_threads(8)
with TemporaryDirectory() as model_workspace:
    patched_model = Path(model_workspace)
    weights_sha256 = prepare_checkpoint(MODEL_DIR, patched_model)
    encoder = SentenceTransformer(str(patched_model), device="cpu", local_files_only=True, trust_remote_code=False)
    tokenizer = encoder.tokenizer
    effective_tokenizer = {
        "class": type(tokenizer).__name__,
        "backend": type(tokenizer.backend_tokenizer.model).__name__,
        "max_seq_length": encoder.max_seq_length,
        "pooling": encoder[1].get_config_dict(),
    }
    expected = {
        "class": "TokenizersBackend", "backend": "Unigram", "max_seq_length": 128,
        "pooling": {"embedding_dimension": 384, "pooling_mode": "mean", "include_prompt": True},
    }
    if effective_tokenizer != expected:
        raise ValueError(f"MiniLM cargó otra configuración: {effective_tokenizer}")
    vectors = encoder.encode(unique.tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    del encoder

if vectors.shape != (len(unique), 384) or not np.isfinite(vectors).all():
    raise ValueError("Dimensiones o valores no válidos en los embeddings.")
features = _full_features(*np.split(vectors[codes], 2))
x_train, x_test = features[:len(train)], features[len(train):]

## Entrenamiento y voto

Ajustamos dos clasificadores con la misma representación. El **MLP** aprende interacciones entre características con una capa oculta de 256 unidades y parada temprana. **XGBoost** usa 500 rondas de boosting, profundidad máxima 3 y tasa de aprendizaje 0,05.

Para cada clase promediamos al 50 % las probabilidades de los dos modelos. Elegimos la clase con mayor probabilidad media. La evaluación local usa el protocolo agrupado de abajo. Para generar el CSV, los dos clasificadores se ajustan con las 12.120 filas etiquetadas, semilla 42 y los parámetros guardados en `protocol.json`. En este notebook usamos CPU.


In [ ]:
SEED = 42
probabilities = []
configs = {}
families = {"mlp_256_alpha0.0003": "mlp", "xgb_d3_500": "xgboost"}

with threadpool_limits(limits=8):
    for tag in protocol["ensemble_2"]:
        family = families[tag]
        params = dict(protocol["configs"][tag])
        if family == "xgboost":
            params["device"] = "cpu"
        configs[tag] = params
        print(f"Ajustando {tag}, semilla {SEED}", flush=True)
        model = build_classifier(family, random_state=SEED, **params)
        model.fit(x_train, train.label.to_numpy())
        np.testing.assert_array_equal(model.classes_, [0, 1, 2])
        proba = model.predict_proba(x_test)
        if proba.shape != (TEST_ROWS, 3) or not np.isfinite(proba).all() or (proba < 0).any():
            raise ValueError(f"Probabilidades inválidas en {tag}.")
        np.testing.assert_allclose(proba.sum(axis=1), 1, atol=1e-5)
        probabilities.append(proba)

predictions = np.mean(probabilities, axis=0).argmax(axis=1)


## Evaluación local guardada

En la ejecución de validación se usaron 9.696 pares de desarrollo. Para cada semilla (42, 7 y 2024) se hicieron cinco particiones estratificadas y agrupadas por el texto exacto de la **premisa**. Cada par se predijo con modelos ajustados en los otros cuatro folds. La auditoría registró cero premisas compartidas entre ajuste y validación en las 15 particiones.

Calculamos la proporción de aciertos (**accuracy**) y el promedio del F1 de las tres clases con el mismo peso (**macro-F1**). La tabla promedia los cinco folds de cada semilla y, en la última fila, las tres semillas. Muestra resultados archivados, sin volver a ajustar los modelos. Son métricas locales. El test oficial se usa aparte para generar el CSV.

Origen en el repositorio: `results_dense_generalization/grouped_seeds.csv` y `split_audit.csv`. Las predicciones de validación por ID están en `grouped_oof_<semilla>.csv.gz`.


In [1]:
import pandas as pd

# Valores archivados del voto MLP + XGBoost, uno por semilla.
validacion = pd.DataFrame([
    (42, 0.62056827281761628, 0.61673718267023481),
    (7, 0.6229345651457765, 0.61949126142392508),
    (2024, 0.62066760290659939, 0.61891907022285286),
], columns=["Semilla", "Accuracy", "Macro-F1"]).set_index("Semilla")
validacion.loc["Media"] = validacion.mean()
(validacion * 100).round(2).rename(columns={"Accuracy": "Accuracy (%)", "Macro-F1": "Macro-F1 (%)"})


,Accuracy (%),Macro-F1 (%)
Semilla,,
42,62.06,61.67
7,62.29,61.95
2024,62.07,61.89
Media,62.14,61.84


## CSV para Kaggle

Guardamos exactamente las columnas `id` y `prediction`. Comprobamos desde disco las 5.195 filas, los IDs únicos en el orden de `sample_submission.csv` y las etiquetas válidas (0, 1 y 2). La salida muestra las primeras filas y la distribución de clases. El manifiesto `minilm_run.json` registra la configuración y el hash del CSV.

Conserva la versión ejecutada del notebook que generó el CSV concreto que vaya a enviar el equipo.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with TemporaryDirectory(dir=OUTPUT_DIR) as staging:
    candidate = Path(staging) / "submission.csv"
    write_submission(test.id, predictions, candidate)
    validate_submission(candidate, DATA_DIR / "sample_submission.csv")
    output = candidate.replace(OUTPUT_DIR / "submission.csv")

record = {
    "seed": SEED,
    "train_rows": len(train),
    "test_rows": len(test),
    "model": protocol["encoder"],
    "features": protocol["features"],
    "tokenizer": effective_tokenizer,
    "tokenizer_config_sha256": FIXED_TOKENIZER,
    "weights_sha256": weights_sha256,
    "configs": configs,
    "versions": {pkg: version(pkg) for pkg in ("torch", "transformers", "tokenizers", "sentence-transformers",
                                              "numpy", "pandas", "scipy", "scikit-learn", "xgboost")},
    "submission_sha256_lf": hashlib.sha256(output.read_bytes().replace(b"\r\n", b"\n")).hexdigest(),
}
(OUTPUT_DIR / "minilm_run.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
print(f"CSV válido: {len(predictions)} filas con IDs en el orden oficial. Salida: {output}")
from IPython.display import display

saved = pd.read_csv(output, dtype={"id": str})
display(saved.head())
print("Recuento de predicciones por clase:", saved.prediction.value_counts().sort_index().to_dict())


### Salida conservada de la ejecución completa anterior

La [ejecución corregida de Kaggle](https://www.kaggle.com/code/jlcabrera/0-aidea-minilm-tokenizer-corregido) (versión 2, semilla 42) generó un CSV de **5.195 filas**. Su fichero descargado se verificó contra el orden de la plantilla. Estas filas y recuentos proceden de ese archivo. El resultado del CSV que envíe el equipo se debe comprobar en su propia versión ejecutada.

| id | prediction |
|---|---:|
| `c6d58c3f69` | 2 |
| `cefcc82292` | 2 |
| `e98005252c` | 0 |
| `58518c10ba` | 1 |
| `c32b0d16df` | 1 |

Recuento de la ejecución archivada por clase: **0 → 2030**, **1 → 1528**, **2 → 1637**.
SHA‑256 del CSV con saltos LF: `07b568d78217a5620a09c0d8d10b418b8b1079a997df33781c4be0422f30bfe9`.
